In [1]:
import warnings
warnings.simplefilter(action='ignore')

import argparse
import os
import random,numpy
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [2]:
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.utils as vutils
import torchvision.transforms.functional as RF
from torchvision.models.feature_extraction import create_feature_extractor
from PIL import Image
import numpy as np
import random,cv2,pandas,os,numpy
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from sklearn.preprocessing import LabelEncoder

In [3]:
seed = 999
print("Random Seed: ", seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

nz = 100
beta1 = 0.5
lr = 0.0001
batch_size=100
ngpu=1
ngf,nc = 3,3
ndf = 64

device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

Random Seed:  999


In [4]:
train = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/train.csv').drop(columns=['ID','efs_time'])
test = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv').drop(columns=['ID'])

In [5]:
encoder = LabelEncoder()

for i in zip(train.columns,train.dtypes):
    if i[1]=='O':
        train[i[0]]=encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1))
    else:
        train[i[0]]=train[i[0]]
for i in zip(test.columns,test.dtypes):
    if i[1]=='O':
        test[i[0]]=numpy.array(encoder.fit_transform(test[i[0]].to_numpy().reshape(-1,1)))
    else:
        test[i[0]]=numpy.array(test[i[0]])

In [6]:
train = train.fillna(0)
test = test.fillna(0)

train_y = train['efs']
train_x = train.drop(columns=['efs'])

for i in test:
    test[i]=test[i].to_numpy()

In [7]:
x_ = torch.from_numpy(train_x.to_numpy()).type('torch.FloatTensor')
y_ = torch.from_numpy(train_y.to_numpy()).type('torch.FloatTensor')
sub = torch.from_numpy(test.to_numpy()).type('torch.FloatTensor')

In [8]:
train_x = torch.utils.data.DataLoader(x_,batch_size=batch_size)
train_y = torch.utils.data.DataLoader(y_,batch_size=batch_size)
test_sub = torch.utils.data.DataLoader(sub,batch_size=batch_size)

In [9]:
class EffnetModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        effnet = torchvision.models.efficientnet_v2_m(weights=torchvision.models.efficientnet.EfficientNet_V2_M_Weights.DEFAULT)
        self.model = create_feature_extractor(effnet, ['flatten'])
        self.nn_fracture = torch.nn.Sequential(
            nn.Linear(1280, 2)
            #nn.Sigmoid()
        )
    def forward(self, x):
        x = torch.Tensor([list(torch.cat((i.view(-1), torch.Tensor([0,0,0]).to(device)), 0)) for i in x])
        m = nn.Upsample(scale_factor=2, mode='bilinear')
        x = m(m(m(m(x.reshape(len(x), 3, 4, 5)))))
        x = self.model(x.to(device))['flatten']
        x = self.nn_fracture(x)
        return x

In [10]:
EFF_NET = EffnetModel().float()
EFF_NET= nn.DataParallel(EFF_NET).to(device)
#EFF_NET.apply(weights_init)

Downloading: "https://download.pytorch.org/models/efficientnet_v2_m-dc08266a.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_m-dc08266a.pth
100%|██████████| 208M/208M [00:01<00:00, 215MB/s] 


In [11]:
criterion = nn.BCEWithLogitsLoss()

optimizer = optim.AdamW(EFF_NET.parameters(), lr=lr,betas=(beta1, 0.999))
scheduler=torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.86)

In [12]:
high_rf,i,j_r,k,z_w_=100000,0,0,0,[]
    
correct,total=0,0

while(True):
    optimizer.zero_grad()
    for data,label in zip(train_x,train_y):
        label=torch.from_numpy(numpy.array([[0,1] if i==1 else [1,0] for i in label])).float().to(device)
        output = EFF_NET(data.to(device))
        err_real = criterion(output, label.to(device))
        err_real.backward()
        optimizer.step()
        #print(err_real.item())
    
    print(f"EPOCH : {i} LOSS_wl : {err_real.item()}")

    if len(z_w_)>=3:
        if len([True for i in range(1,4) if z_w_[len(z_w_)-i]<=z_w_[len(z_w_)-3] and z_w_[len(z_w_)-i]>=z_w_[len(z_w_)-4]])==3:
            z_w_,j_r=[],0
            print(f"lr_br:{optimizer.param_groups[0]['lr']}".upper())
            scheduler.step()
            print(f"lr_up:{optimizer.param_groups[0]['lr']}".upper())
            
    z_w_.append(err_real.item())
    if err_real.item()<high_rf:
            high_rf=err_real.item()
            torch.save(EFF_NET.state_dict(),f'/kaggle/working/{err_real.item()}.pth')
    if i==200:
        break
    i+=1

EPOCH : 0 LOSS_wl : 0.6373664140701294
EPOCH : 1 LOSS_wl : 0.6489258408546448
EPOCH : 2 LOSS_wl : 0.6684688925743103
EPOCH : 3 LOSS_wl : 0.6713695526123047
EPOCH : 4 LOSS_wl : 0.6804060339927673
EPOCH : 5 LOSS_wl : 0.6525244116783142
EPOCH : 6 LOSS_wl : 0.642271876335144
EPOCH : 7 LOSS_wl : 0.644968569278717
EPOCH : 8 LOSS_wl : 0.6289097666740417
EPOCH : 9 LOSS_wl : 0.6537116765975952
EPOCH : 10 LOSS_wl : 0.6276618242263794
EPOCH : 11 LOSS_wl : 0.6612165570259094
EPOCH : 12 LOSS_wl : 0.6317316889762878
EPOCH : 13 LOSS_wl : 0.6548147201538086



KeyboardInterrupt

